In [1]:
import torch
import numpy as np

import training_models
from rejector import Rejector
import training
from ensemble import TorchRejectionEnsemble
from DMTimeShardDataset import DMTimeShardDataset
import plotly.graph_objects as go

from torch.utils.data import DataLoader, TensorDataset, Subset
from rejection_ensemble_helper import _extract_labels, plot_snr_distributions, eval_optimal, _strip_prefix_from_state_dict, _maybe_extract_state_dict
import math

from skrejector import SNRDT_Rejector

In [ ]:
device = "cuda"

#rejector_path = "./rejector2.pth"
#rejector_path = "/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/checkpoints/ch_point_DM_time_binary_classificator_241002_3_dropout_256/prot-015-0.980-0.985.pth"
rejector_path = "/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/rejector_checkpoints/prot-rejector_test-011-0.544-0.510.pth"
rejector_checkpoint = torch.load(rejector_path, map_location=device)
rejector_state_dict = _strip_prefix_from_state_dict(_maybe_extract_state_dict(rejector_checkpoint))
r = training_models.models_htable["DM_time_binary_classificator_241002_3"](256, mode="dmt", dropout=False, device=device).to(device)
r.load_state_dict(rejector_state_dict)

#freeze first n-1 layers
for p in r.parameters():
    p.requires_grad = False

for p in r.fc2.parameters():
    p.requires_grad = True
    
rejector = Rejector(r, device)

KeyboardInterrupt: 

In [2]:
device = "cuda"

rejector = SNRDT_Rejector(device)

In [3]:
small_weights = "/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/checkpoints/ch_point_DM_time_binary_classificator_241002_3_dropout_256/prot-015-0.980-0.985.pth"

small_model = training_models.models_htable["DM_time_binary_classificator_241002_3"](256, mode="dmt", dropout=True, device=device).to(device)
small_model.load_state_dict(torch.load(small_weights, map_location=device)["model_state_dict"])


big_weights = "/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/checkpoints/ch_point_DM_time_binary_classificator_resnet18_dropout_256/prot-014-0.972-0.977.pth"

big_model = training_models.models_htable['DM_time_binary_classificator_resnet18'](256, mode="ft", dropout=True, device=device).to(device)
big_model.load_state_dict(torch.load(big_weights, map_location=device)["model_state_dict"])


rejection_ensemble = TorchRejectionEnsemble(small_model, big_model, p=0.8, rejector=rejector, calibration=False)

In [6]:
dataset_cfg = {
        "output_dir": "/cephfs/users/oleksjuk/MA/WP2-1/DM_time_dataset_creator/outputs",
        "prefix": "B0531+21_59000_48386",
    }

full_train_dataset = DMTimeShardDataset(dataset_cfg, use_freq_time=True, split="train")
full_train_dataset.labels = training.label_encoding(full_train_dataset.labels.astype(object))


val_fraction = 0.1111
#val_fraction = 0.95
if not 0 < val_fraction < 1:
    raise ValueError("'val_fraction' must be between 0 and 1.")

num_train_samples = len(full_train_dataset)
if num_train_samples < 2:
    raise ValueError("Need at least 2 training samples to create a validation split.")

split_idx_val = math.floor(num_train_samples * (1 - val_fraction))
split_idx_val = min(max(split_idx_val, 1), num_train_samples - 1)

train_indices = range(0, split_idx_val)
val_indices = range(split_idx_val, num_train_samples)

pulse_train_dataset = Subset(full_train_dataset, train_indices)
val_dataset = Subset(full_train_dataset, val_indices)

#train_dataset = full_train_dataset

pulse_test_dataset = DMTimeShardDataset(dataset_cfg, use_freq_time=True, split="test")
pulse_test_dataset.labels = training.label_encoding(pulse_test_dataset.labels.astype(object))

pulse_train_loader = DataLoader(pulse_train_dataset,
                          batch_size=256,
                          shuffle=False,
                          num_workers=0)
#pulse_val_loader = DataLoader(val_dataset,
#                          batch_size=256,
#                          shuffle=False,
#                          num_workers=8)
pulse_test_loader = DataLoader(pulse_test_dataset,
                        batch_size=256,
                        shuffle=False,
                        num_workers=0)

print("Train Loader length: ", len(pulse_train_loader))
#print("Val Loader length: ", len(pulse_val_loader))
print("Test Loader length: ", len(pulse_test_loader))
print("train dataset length: ", len(pulse_train_dataset))
#print("val dataset length: ", len(pulse_val_dataset))
print("test dataset length: ", len(pulse_test_dataset))


Converting string labels to numeric: ['Artefact' 'Pulse'] -> [0, 1]
Converting string labels to numeric: ['Artefact' 'Pulse'] -> [0, 1]
Train Loader length:  5383
Test Loader length:  673
train dataset length:  1377866
test dataset length:  172288
Converting string labels to numeric: ['Artefact' 'Pulse'] -> [0, 1]
Train Loader length:  5383
Test Loader length:  673
train dataset length:  1377866
test dataset length:  172288


In [7]:
routing_targets_train, routing_targets_test, pulse_train_preds_small, pulse_test_preds_small = rejection_ensemble.prepare_fit(pulse_train_loader, pulse_test_loader)

get fsmall train predictions
get fbig train predictions
get fbig train predictions
get fsmall test predictions
get fsmall test predictions
get fbig test predictions
get fbig test predictions
start fitting rejector
start fitting rejector
TRAIN: targets before balancing. REJECT:  25535 no REJECT:  1352331
TRAIN: targets after balancing. REJECT:  25535 no REJECT:  1352331
TEST: targets before balancing. REJECT:  25535 no REJECT:  1352331
TEST: targets after balancing. REJECT:  25535 no REJECT:  1352331
created pseudo labels
TRAIN: targets before balancing. REJECT:  25535 no REJECT:  1352331
TRAIN: targets after balancing. REJECT:  25535 no REJECT:  1352331
TEST: targets before balancing. REJECT:  25535 no REJECT:  1352331
TEST: targets after balancing. REJECT:  25535 no REJECT:  1352331
created pseudo labels


In [5]:
print(pulse_train_preds_small.shape)

(77504, 2)


In [8]:
balanced_pulse_train_loader, balanced_routing_train_targets = rejection_ensemble._splitTrainData(pulse_train_loader, routing_targets_train, pulse_train_preds_small)
balanced_pulse_test_loader, balanced_routing_test_targets = rejection_ensemble._splitTrainData(pulse_test_loader, routing_targets_test, pulse_test_preds_small)

targets before balancing. REJECT:  25535 no REJECT:  1352331
targets after balancing. REJECT:  25535 no REJECT:  25535
targets before balancing. REJECT:  33468 no REJECT:  138820
targets after balancing. REJECT:  33468 no REJECT:  33468
targets after balancing. REJECT:  25535 no REJECT:  25535
targets before balancing. REJECT:  33468 no REJECT:  138820
targets after balancing. REJECT:  33468 no REJECT:  33468


In [18]:
splits_path = "./dm_time_splits.pth"
rejection_ensemble.save_balanced_splits(
    splits_path,
    pulse_train_loader,
    routing_targets_train,
    pulse_test_loader,
    routing_targets_test,
)
print(f"Saved unbalanced loaders and targets to {splits_path}")

Saved unbalanced loaders and targets to ./dm_time_splits.pth


In [19]:
balanced_splits_path = "./balanced_dm_time_splits.pth"
rejection_ensemble.save_balanced_splits(
    balanced_splits_path,
    balanced_pulse_train_loader,
    balanced_routing_train_targets,
    balanced_pulse_test_loader,
    balanced_routing_test_targets,
 )
print(f"Saved balanced loaders and targets to {balanced_splits_path}")

Saved balanced loaders and targets to ./balanced_dm_time_splits.pth


In [4]:
(
    cached_balanced_pulse_train_loader,
    cached_balanced_routing_train_targets,
    cached_balanced_pulse_test_loader,
    cached_balanced_routing_test_targets,
 ) = rejection_ensemble.load_balanced_splits("./balanced_dm_time_splits.pth")

print(
    f"Reloaded {len(cached_balanced_pulse_train_loader.dataset)} training samples and {len(cached_balanced_pulse_test_loader.dataset)} test samples.",
    f"Training targets shape: {cached_balanced_routing_train_targets.shape}",
    f"Test targets shape: {cached_balanced_routing_test_targets.shape}",
    sep="\n",
 )

Converting string labels to numeric: ['Artefact' 'Pulse'] -> [0, 1]
Converting string labels to numeric: ['Artefact' 'Pulse'] -> [0, 1]
Reloaded 51070 training samples and 66936 test samples.
Training targets shape: (51070,)
Test targets shape: (66936,)


In [5]:
(
    cached_pulse_train_loader,
    cached_routing_train_targets,
    cached_pulse_test_loader,
    cached_routing_test_targets,
) = rejection_ensemble.load_balanced_splits("./dm_time_splits.pth")

train_samples = len(cached_pulse_train_loader.dataset)
train_targets = cached_routing_train_targets.shape[0]
test_samples = len(cached_pulse_test_loader.dataset)
test_targets = cached_routing_test_targets.shape[0]

print(
    f"Reloaded {train_samples} training samples and {test_samples} test samples.",
    f"Training targets shape: {cached_routing_train_targets.shape}",
    f"Test targets shape: {cached_routing_test_targets.shape}",
    sep="\n",
)

if train_samples != train_targets:
    raise ValueError(
        f"Train loader/sample mismatch: {train_samples} samples vs {train_targets} routing targets. "
        "Regenerate the split to keep them in sync."
    )
if test_samples != test_targets:
    raise ValueError(
        f"Test loader/sample mismatch: {test_samples} samples vs {test_targets} routing targets."
    )

Converting string labels to numeric: ['Artefact' 'Pulse'] -> [0, 1]
Converting string labels to numeric: ['Artefact' 'Pulse'] -> [0, 1]
Reloaded 1377866 training samples and 172288 test samples.
Training targets shape: (1377866,)
Test targets shape: (172288,)


In [11]:
pulse_train_dataset = pulse_train_loader.dataset
pulse_train_targets = _extract_labels(pulse_train_dataset)

pulse_idx_0 = np.where(pulse_train_targets == 0)[0]
pulse_idx_1 = np.where(pulse_train_targets == 1)[0]
print(len(pulse_idx_0)," ", len(pulse_idx_1))

balanced_routing_idx_0 = np.where(balanced_routing_train_targets == 0)[0]
balanced_routing_idx_1 = np.where(balanced_routing_train_targets == 1)[0]
print(len(balanced_routing_idx_0)," ", len(balanced_routing_idx_1))

routing_idx_0 = np.where(routing_targets_train == 0)[0]
routing_idx_1 = np.where(routing_targets_train == 1)[0]
print(len(routing_idx_0)," ", len(routing_idx_1))

KeyboardInterrupt: 

In [ ]:
plot_snr_distributions(pulse_train_dataset, routing_idx_0, routing_idx_1)

In [ ]:
print('a')
(targets_train, targets_test), fsmall, fbig, rejector = rejection_ensemble.fit(cached_balanced_pulse_train_loader, cached_balanced_routing_train_targets, cached_balanced_pulse_test_loader, cached_balanced_routing_test_targets)

a
start creating pseudo labels


In [ ]:
targets_train = torch.Tensor(targets_train)
unique, counts = torch.unique(targets_train, return_counts=True)
print(unique, counts, counts.float() / counts.sum())

In [ ]:
torch.save(rejection_ensemble.rejector.state_dict(), "./rejector2.pth")

In [8]:
routing_targets_train

array([0, 0, 0, ..., 0, 0, 0], shape=(77504,))

In [8]:
eval_optimal(small_model, big_model, train_dataloader = pulse_train_loader, test_dataloader = pulse_test_loader, optimal_train_routing = routing_targets_train, optimal_test_routing = routing_targets_test)

evaluating model on training data


train acc: 0.9981703590915227; train loss: 0.0075049485026656496
evaluating model on test data
test acc: 0.9540130479197623; test loss: 0.18388451580484058


(0.9981703590915227,
 0.0075049485026656496,
 None,
 None,
 0.9540130479197623,
 0.18388451580484058)

In [9]:
eval_optimal(small_model, small_model, train_dataloader = pulse_train_loader, test_dataloader = pulse_test_loader, optimal_train_routing = routing_targets_train, optimal_test_routing = routing_targets_test)

evaluating model on training data
train acc: 0.9951606324562766; train loss: 0.011582073725804274
evaluating model on test data
test acc: 0.7965267459138187; test loss: 1.2052659331246613


(0.9951606324562766,
 0.011582073725804274,
 None,
 None,
 0.7965267459138187,
 1.2052659331246613)

In [10]:
eval_optimal(big_model, big_model, train_dataloader = pulse_train_loader, test_dataloader = pulse_test_loader, optimal_train_routing = routing_targets_train, optimal_test_routing = routing_targets_test)

evaluating model on training data
train acc: 0.980893642778035; train loss: 0.0604686629931634
evaluating model on test data
test acc: 0.9513489041604755; test loss: 0.14337033880833


(0.980893642778035,
 0.0604686629931634,
 None,
 None,
 0.9513489041604755,
 0.14337033880833)

In [5]:
rejection_ensemble.rejector.fit(cached_pulse_train_loader, cached_routing_train_targets, cached_pulse_test_loader, cached_routing_test_targets)

approx_snr vs TransientX SNR (first 5): approx=[6.4451337 4.181843  4.5439377 4.1534433 4.2075377], tx=[4.4 nan 4.4 4.2 nan], diff=[2.0451336         nan 0.14393759 0.04655647        nan]
approx_snr vs TransientX SNR (first 5): approx=[4.7214355 4.215567  4.283016  4.1111817 4.871322 ], tx=[nan 4.3 4.4 nan 6.4], diff=[       nan 0.08443308 0.11698389        nan 1.5286779 ]
approx_snr vs TransientX SNR (first 5): approx=[4.4179144 4.283016  4.0790634 4.586537  3.9856274], tx=[4.4 4.2 4.7 4.7 4.4], diff=[0.0179143  0.0830164  0.6209364  0.11346292 0.41437268]
approx_snr vs TransientX SNR (first 5): approx=[4.2075377 4.519088  4.4179144 4.614937  5.182929 ], tx=[4.4 4.3 4.3 nan 4.8], diff=[0.19246244 0.2190876  0.1179142         nan 0.38292885]
approx_snr vs TransientX SNR (first 5): approx=[7.824093  7.6442285 4.2396564 4.4002495 4.1433005], tx=[nan 6.5 nan nan 4.4], diff=[       nan 1.1442285         nan        nan 0.25669956]
approx_snr vs TransientX SNR (first 5): approx=[4.3504653 3.

KeyboardInterrupt: 

In [14]:
rejection_ensemble.rejector.snr_threshold_

5.791032552719116

In [ ]:
rejection_ensemble.rejector.eval(cached_balanced_pulse_train_loader, cached_balanced_routing_train_targets, cached_pulse_test_loader, cached_routing_test_targets)

generate SNR labels for test data
evaluate test data
SNRDTRejector Test Accuracy: 0.7480323644130757 (X: SNR; Y: routing labels)
evaluate test data
SNRDTRejector Test Accuracy: 0.7480323644130757 (X: SNR; Y: routing labels)


0.7480323644130757

In [17]:
rejection_ensemble.eval(train_dataloader=pulse_test_loader, test_dataloader=pulse_test_loader)

TypeError: SNRDT_Rejector.eval() missing 2 required positional arguments: 'pulse_X_train_dataloader' and 'routing_targets_train'